<a href="https://colab.research.google.com/github/thuviettran/demo-github/blob/main/movie_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Hybrid Movie Recommender tối ưu cho MovieLens 1M
# Bao gồm: re-index ID, chuẩn hóa rating, genre embedding, early stopping

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import (
    Input, Embedding, Flatten, Dense, Concatenate,
    Dropout, GlobalAveragePooling1D
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

# =====================
# 1. Load dữ liệu .dat
# =====================
ratings = pd.read_csv(
    "ratings.dat",
    sep="::",
    engine="python",
    names=["userId", "movieId", "rating", "timestamp"]
)

movies = pd.read_csv(
    "movies.dat",
    sep="::",
    engine="python",
    names=["movieId", "title", "genres"],
    encoding="latin-1"
)

# =====================
# 2. Re-index ID (quan trọng)
# Fit encoder trên TẬP HỢP movieId của cả ratings + movies để tránh unseen labels
# =====================
user_encoder = LabelEncoder()
movie_encoder = LabelEncoder()

ratings["userId"] = user_encoder.fit_transform(ratings["userId"])

# Fit movie encoder trên union movieId
all_movie_ids = pd.concat([ratings["movieId"], movies["movieId"]]).unique()
movie_encoder.fit(all_movie_ids)

ratings["movieId"] = movie_encoder.transform(ratings["movieId"])
movies["movieId"] = movie_encoder.transform(movies["movieId"])

num_users = ratings["userId"].nunique()
num_movies = len(all_movie_ids)

# =====================
# 3. Xử lý genre → danh sách index
# =====================
movies["genre_list"] = movies["genres"].apply(lambda x: x.split("|"))

all_genres = sorted({g for sub in movies["genre_list"] for g in sub})
genre_to_idx = {g: i + 1 for i, g in enumerate(all_genres)}  # 0 = padding

max_genres = movies["genre_list"].apply(len).max()
num_genres = len(all_genres) + 1


def encode_genres(genre_list):
    idxs = [genre_to_idx[g] for g in genre_list]
    return idxs + [0] * (max_genres - len(idxs))


movies["genre_encoded"] = movies["genre_list"].apply(encode_genres)

# =====================
# 4. Merge dữ liệu
# =====================
df = ratings.merge(movies[["movieId", "genre_encoded"]], on="movieId")

# =====================

# Sửa lại phần 5 & 6
y = df["rating"].values.astype("float32")
user_input = df["userId"].values
movie_input = df["movieId"].values
genre_input = np.stack(df["genre_encoded"].values)

# Chia Train/Test TRƯỚC
(
    user_train, user_test,
    movie_train, movie_test,
    genre_train, genre_test,
    y_train_raw, y_test_raw
) = train_test_split(
    user_input, movie_input, genre_input, y,
    test_size=0.2,
    random_state=42
)

# Chuẩn hóa SAU KHI chia (chỉ dùng tham số của tập Train)
y_mean = y_train_raw.mean()
y_std = y_train_raw.std()

y_train = (y_train_raw - y_mean) / y_std
y_test = (y_test_raw - y_mean) / y_std # Chuẩn hóa tập test bằng mean/std của tập train
# =====================
# 7. Xây dựng Hybrid Model tối ưu
# =====================
embedding_dim = 64

user_in = Input(shape=(1,))
movie_in = Input(shape=(1,))
genre_in = Input(shape=(max_genres,))

# User & Movie embedding (Collaborative Filtering)
user_emb = Embedding(num_users, embedding_dim)(user_in)
movie_emb = Embedding(num_movies, embedding_dim)(movie_in)

user_vec = Flatten()(user_emb)
movie_vec = Flatten()(movie_emb)

# Genre embedding (Content-based)
genre_emb = Embedding(num_genres, 32, mask_zero=True)(genre_in)
genre_vec = GlobalAveragePooling1D()(genre_emb)

# Combine tất cả
x = Concatenate()([user_vec, movie_vec, genre_vec])

# Deep layers
x = Dense(256, activation="relu")(x)
x = Dropout(0.4)(x)
x = Dense(128, activation="relu")(x)
x = Dense(64, activation="relu")(x)

out = Dense(1, activation="linear")(x)

model = Model(inputs=[user_in, movie_in, genre_in], outputs=out)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
    metrics=["mae"]
)

model.summary()

# =====================
# 8. Train với EarlyStopping
# =====================
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    [user_train, movie_train, genre_train],
    y_train,
    validation_split=0.1,
    epochs=20,
    batch_size=512,
    callbacks=[early_stop],
    verbose=1
)

# =====================
# 9. Evaluate
# =====================
loss, mae = model.evaluate(
    [user_test, movie_test, genre_test],
    y_test,
    verbose=0
)

rmse = np.sqrt(loss)

print(f"Test RMSE: {rmse:.4f}")
print(f"Test MAE: {mae:.4f}")

# =====================
# =====================
# Lấy giá trị tệp test và so sánh dự đoán
# =====================

# 1. Yêu cầu mô hình dự đoán trên tập test
y_pred_normalized = model.predict([user_test, movie_test, genre_test], verbose=0)

# 2. Scale ngược giá trị dự đoán về lại thang điểm 1-5 sao
y_pred_real = (y_pred_normalized.flatten() * y_std) + y_mean

# 3. Lấy giá trị thực tế (chính là y_test_raw ở bước chia dữ liệu)
y_true_real = y_test_raw

# 4. Đưa tất cả vào một DataFrame để dễ quan sát
test_results_df = pd.DataFrame({
    'User_ID_Encoded': user_test,
    'Movie_ID_Encoded': movie_test,
    'True_Rating': y_true_real,
    'Predicted_Rating': np.round(y_pred_real, 2), # Làm tròn 2 chữ số thập phân
    'Sai_Số': np.round(abs(y_true_real - y_pred_real), 2) # Tính độ lệch tuyệt đối
})

# Hiển thị 10 dòng đầu tiên của tập test
print("--- 10 GIAO DỊCH TRONG TẬP TEST ---")
print(test_results_df.head(200))
# 10. Hàm gợi ý phim
# =====================
movies_full = movies.copy()


def recommend(user_id_raw, top_k=10):
    user_id = user_encoder.transform([user_id_raw])[0]

    user_arr = np.full(len(movies_full), user_id)
    movie_arr = movies_full["movieId"].values
    genre_arr = np.stack(movies_full["genre_encoded"].values)

    preds = model.predict([user_arr, movie_arr, genre_arr], verbose=0).flatten()

    # scale ngược rating
    preds = preds * y_std + y_mean

    movies_full["pred_rating"] = preds

    return movies_full.sort_values("pred_rating", ascending=False).head(top_k)[
        ["title", "genres", "pred_rating"]
    ]


# Ví dụ gợi ý cho user gốc = 1
print(recommend(1))


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 6)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 64)     │    386,560 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1, 64)     │    248,512 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 6, 32)     │        608 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 6)         │          0 │ input_layer_2[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 64)        │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 64)        │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ embedding_2[0][0… │
│ (GlobalAveragePool… │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 160)       │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0],  │
│                     │                   │            │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │     41,216 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     32,896 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      8,256 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         65 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 718,113 (2.74 MB)

 Trainable params: 718,113 (2.74 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 0.7420 - mae: 0.6854 - val_loss: 0.6501 - val_mae: 0.6358
Epoch 2/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 0.6340 - mae: 0.6276 - val_loss: 0.6237 - val_mae: 0.6238
Epoch 3/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.6005 - mae: 0.6110 - val_loss: 0.6095 - val_mae: 0.6167
Epoch 4/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.5719 - mae: 0.5963 - val_loss: 0.6021 - val_mae: 0.6098
Epoch 5/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 0.5515 - mae: 0.5851 - val_loss: 0.6003 - val_mae: 0.6135
Epoch 6/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.5354 - mae: 0.5769 - val_loss: 0.5966 - val_mae: 0.6077
Epoch 7/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.5204 - mae: 0.5683 - val_loss: 0.5962 - val_mae: 0.6076
Epoch 8/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.5051 - mae: 0.5596 - val_loss: 0.5974 - val_mae: 0.6077
Epoch 9/20
1407/1407 ━━━━━━━━━━━━━━━━━━

In [ ]:
import pickle

# Lưu mô hình Deep Learning
model.save("movie_recommender.keras")

# Lưu các bộ mã hóa (Encoder) và dữ liệu cần thiết để phục vụ Web
artifacts = {
    "user_encoder": user_encoder,
    "movies_full": movies, # Lưu lại thông tin phim
    "y_mean": y_mean,
    "y_std": y_std
}

with open("artifacts.pkl", "wb") as f:
    pickle.dump(artifacts, f)

print("Đã lưu model (movie_recommender.h5) và artifacts (artifacts.pkl) thành công!")

Đã lưu model (movie_recommender.h5) và artifacts (artifacts.pkl) thành công!


In [ ]:
print(test_results_df.head(4000))

      User_ID_Encoded  Movie_ID_Encoded  True_Rating  Predicted_Rating  Sai_Số
0                5411              2614          2.0              3.84    1.84
1                5439               892          5.0              4.84    0.16
2                 367              3648          4.0              3.42    0.58
3                 424              1672          4.0              3.17    0.83
4                4941              3628          1.0              2.59    1.59
...               ...               ...          ...               ...     ...
3995             3768              2031          4.0              4.26    0.26
3996             4336              1021          1.0              4.33    3.33
3997             3681              2222          4.0              4.33    0.33
3998             5584              3827          5.0              4.31    0.69
3999             4866              3026          4.0              4.36    0.36

[4000 rows x 5 columns]


In [ ]:
# Xuất ra file Excel
test_results_df.to_excel("Ket_Qua_Tap_Test.xlsx", index=False)

print("Đã xuất thành công ra file Ket_Qua_Tap_Test.xlsx!")

Đã xuất thành công ra file Ket_Qua_Tap_Test.xlsx!
